In [ ]:
!pip install timm -q

import os, random, sys, time
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

print(f"torch={torch.__version__}, cuda={torch.cuda.is_available()}, timm={timm.__version__}")

In [ ]:
DS_DIR   = Path('/kaggle/input/birdclef2026-effnet-multiwindow')
WORK     = Path('/kaggle/working')
LABELS_NPZ = DS_DIR / 'labels_v2.npz'
CACHE_NPY  = DS_DIR / 'cache_v2.npy'
CACHE_META = DS_DIR / 'cache_v2_meta.npz'
N_FOLDS=5; N_EPOCHS=30; BATCH_SIZE=32; LR=5e-4; EFFNET_DIM=1280
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    try:
        _x = torch.randn(4, 4).cuda()
        _y = (_x * 2.0).sum()
        print(f"GPU basic op OK (cap={torch.cuda.get_device_capability()}, name={torch.cuda.get_device_name()})")
        del _x, _y
    except Exception as _e:
        print(f"GPU basic op FAILED: {_e}")
        print("Falling back to CPU")
        DEVICE = torch.device('cpu')

print(f"cache: {CACHE_NPY}")
print(f"labels: {LABELS_NPZ}")

In [ ]:
class BirdCacheDataset(Dataset):
    def __init__(self, sample_indices, labels, window_lists, cache, is_train=True):
        self.sample_indices=sample_indices; self.labels=labels
        self.window_lists=window_lists; self.cache=cache; self.is_train=is_train
    def __len__(self): return len(self.sample_indices)
    def __getitem__(self, idx):
        wl=self.window_lists[idx]
        w=random.choice(wl) if (self.is_train and len(wl)>1) else wl[0]
        return torch.from_numpy(self.cache[w].astype(np.float32)), \
               torch.from_numpy(np.asarray(self.labels[idx],dtype=np.float32))

def cpu_augment(x, y, freq_mask=30, time_mask=40, mixup_alpha=1.0, mixup_theta=0.8):
    """All augmentation on CPU tensors to avoid CUDA kernel compatibility issues."""
    F, T = x.shape[-2], x.shape[-1]
    for _ in range(2):
        f0 = random.randint(0, max(0, F - freq_mask))
        x[..., f0:f0+freq_mask, :] = 0
    for _ in range(2):
        t0 = random.randint(0, max(0, T - time_mask))
        x[..., :, t0:t0+time_mask] = 0
    if random.random() < mixup_theta:
        lam = float(np.random.beta(mixup_alpha, mixup_alpha))
        idx = torch.randperm(x.size(0))
        x = lam*x + (1-lam)*x[idx]
        y = lam*y + (1-lam)*y[idx]
    return x, y

class EffNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.stem_conv = nn.Conv2d(1, 3, 3, 1, 1, bias=False)
        self.backbone = timm.create_model('tf_efficientnetv2_b0', pretrained=True, in_chans=3, num_classes=0)
        self.head = nn.Linear(EFFNET_DIM, n_classes)
    def forward(self, x):
        return self.head(self.backbone(self.stem_conv(x)))

def set_seed(s=42): random.seed(s); np.random.seed(s); torch.manual_seed(s)
print("Classes ready")

In [ ]:
def train_fold(fold, ld, cache, meta):
    set_seed(42+fold)
    n=len(ld['paths'])
    wl=[[] for _ in range(n)]
    for w,si in enumerate(meta['sample_idx']): wl[int(si)].append(w)
    labels=ld['labels'].astype(np.float32)
    source=np.array([str(s) for s in ld['source']])
    pstrat=np.array([str(s) for s in ld['primary_strat']])
    fi=np.where(source=='focal')[0]; si=np.where(source!='focal')[0]
    skf=StratifiedKFold(N_FOLDS,shuffle=True,random_state=42)
    ftr,fval=list(skf.split(fi,pstrat[fi]))[fold]
    tr_idx=np.concatenate([fi[ftr],si]); va_idx=fi[fval]
    tr_ds=BirdCacheDataset(tr_idx,labels[tr_idx],[wl[i] for i in tr_idx],cache,True)
    va_ds=BirdCacheDataset(va_idx,labels[va_idx],[wl[i] for i in va_idx],cache,False)
    tr_ld=DataLoader(tr_ds,BATCH_SIZE,shuffle=True,num_workers=4,pin_memory=True,drop_last=True)
    va_ld=DataLoader(va_ds,BATCH_SIZE,shuffle=False,num_workers=4)
    n_cls=labels.shape[1]
    print(f"\n{'='*50}\nFold {fold+1}: Train={len(tr_ds)}(f={len(ftr)},ss={len(si)}),Val={len(va_ds)}")
    model=EffNet(n_cls).to(DEVICE)
    crit=nn.BCEWithLogitsLoss()
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=N_EPOCHS,eta_min=1e-6)
    cp=WORK/f'ckpt_f{fold}.pth'; ep0=0; best=0.0
    if cp.exists():
        ck=torch.load(cp,map_location=DEVICE,weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['optimizer'])
        sch.load_state_dict(ck['scheduler']); ep0=ck['epoch']+1; best=ck['best']
    for ep in range(ep0,N_EPOCHS):
        t0=time.time(); model.train(); tl=0
        for sp,tgt in tr_ld:
            sp,tgt = cpu_augment(sp, tgt)  # augmentation on CPU
            sp=sp.to(DEVICE); tgt=tgt.to(DEVICE)
            lg=model(sp); loss=crit(lg,tgt)
            opt.zero_grad(); loss.backward(); opt.step(); tl+=loss.item()
        sch.step()
        model.eval(); pa,ta=[],[]
        with torch.no_grad():
            for sp,tgt in va_ld:
                lg=model(sp.to(DEVICE)); pa.append(torch.sigmoid(lg).cpu().numpy()); ta.append(tgt.numpy())
        p=np.vstack(pa); t=(np.vstack(ta)>0.5).astype(np.float32)  # binarize for AUC
        aucs=[roc_auc_score(t[:,j],p[:,j]) for j in range(n_cls) if t[:,j].sum()>0]
        auc=float(np.mean(aucs)) if aucs else 0.
        nb2=max(1,len(tr_ld))
        print(f"  Ep{ep+1}/{N_EPOCHS} loss={tl/nb2:.4f} auc={auc:.4f} ({time.time()-t0:.0f}s)",flush=True)
        if auc>best:
            best=auc; torch.save(model.state_dict(),WORK/f'best_fold{fold}.pth')
            print(f"    -> best {best:.4f}",flush=True)
        torch.save({'epoch':ep,'model':model.state_dict(),'optimizer':opt.state_dict(),
                    'scheduler':sch.state_dict(),'best':best},cp)
    if cp.exists(): cp.unlink()
    print(f"Fold {fold+1} best={best:.4f}"); return best

set_seed(42)
print(f"Loading cache: {CACHE_NPY}")
cache=np.load(str(CACHE_NPY),mmap_mode='r')
meta=np.load(str(CACHE_META),allow_pickle=True)
ld=dict(np.load(str(LABELS_NPZ),allow_pickle=True))
print(f"cache={cache.shape} labels={ld['labels'].shape}")
aucs=[]; tall=time.time()
for fold in range(N_FOLDS):
    bp=WORK/f'best_fold{fold}.pth'; ck=WORK/f'ckpt_f{fold}.pth'
    if bp.exists() and not ck.exists(): print(f"Fold {fold+1} done,skip"); aucs.append(-1.); continue
    aucs.append(train_fold(fold,ld,cache,meta))
done=[a for a in aucs if a>0]
if done:
    for i,a in enumerate(aucs):
        if a>0: print(f"  Fold {i+1}: {a:.4f}")
    print(f"  Mean: {np.mean(done):.4f}")
print(f"Total: {(time.time()-tall)/60:.1f}min")

In [ ]:
print("Output weights:")
for f in sorted(WORK.glob('best_fold*.pth')):
    print(f"  {f.name}  {f.stat().st_size/1e6:.1f} MB")